In [2]:
# 提取案例库数据的标签
import pandas as pd
import json
import random
import re
import os

def read_json_files(folder_path):
    articles,charges,keys,facts,paths,node_ids = [],[],[],[],[],[]
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
                    for line in lines:
                        line = json.loads(line)
                        pattern = r"根据法条第(.+?)条,被告人犯了(.+?)罪"
                        matches = re.search(pattern, line["value"])
                        if matches:
                            article = matches.group(1)
                            charge = matches.group(2)
                            articles.append(article)
                            charges.append(charge)
                            keys.append(line["key2"])
                            facts.append(line["key"])
                            paths.append(line["path_list"])
                            node_ids.append(line["node_id"])
    return facts,keys,charges,articles,paths,node_ids
facts,keys,charges,articles,paths,node_ids = read_json_files("/root/data1/liang/self-correct-retriever/data/hera_knowbase3/类案")
know_df = pd.DataFrame()
know_df["fact"],know_df["key"],know_df["charge"],know_df["article"],know_df["path"],know_df["node_id"]=facts,keys,charges,articles,paths,node_ids
print(len(know_df))
know_df.head(2)


1000


,fact,key,charge,article,path,node_id
0,经审理查明，原审判决书确认的证据，均经一审庭审举证、质证，证据来源合法，内容真实客观，证据之...,"[经审理查明, 原审判决书确认的证据, 一审庭审举证、质证, 证据来源合法, 内容真实客观,...",伪造公司、企业、事业单位、人民团体印章,二百八十零,"[类案, 刑法, 妨害社会管理秩序罪, 伪造公司、企业、事业单位、人民团体印章]","[0, 1000, 2000]"
1,克东县人民检察院指控，2016年1月1日20许，在宏博花园西侧被告人王2某经营的恒信中介公司...,"[聚众, 抽头渔利, 中华人民共和国刑法, 罪, 判处至一年六个月并处罚金]",赌博,三百零三,"[类案, 刑法, 妨害社会管理秩序罪, 赌博]","[1, 1001, 2001]"


In [3]:
# 构造三元组，一个标签相同，一个不同，随机选
import pandas as pd
import json
import random
from tqdm import tqdm
with open("/root/data1/liang/self-correct-retriever/data/knowledge_base/single_train_ljp.json","r") as f:
    data_all = []
    lines = f.readlines()
    random.shuffle(lines)
    for line in tqdm(lines):
        line = json.loads(line)
        charge,article = line["charge"],line["article"]
        same_label_data = know_df[(know_df['charge'] == charge) & (know_df['article'] == article)]
        try:
            random_same_label_data = same_label_data.sample(n=1)
        except:
            continue
        different_label_data = know_df[(know_df['charge'] != charge) | (know_df['article'] != article)]
        try:
            random_different_label_data = different_label_data.sample(n=1)
        except:
            continue
        data_all.append({'origin': line["fact"],
                         'entailment': random_same_label_data.iloc[0]["fact"], 
                         'entailment_key':random_same_label_data.iloc[0]["key"], 
                         'entailment_path':random_same_label_data.iloc[0]["path"],
                         'entailment_node_id':random_same_label_data.iloc[0]["node_id"],
                         'contradiction': random_different_label_data.iloc[0]["fact"],
                         'contradiction_key':random_different_label_data.iloc[0]["key"],
                         'contradiction_path':random_different_label_data.iloc[0]["path"],
                         'contradiction_node_id':random_different_label_data.iloc[0]["node_id"]})
        
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/simcse_test_ljp.json"
with open(write_path, "w") as f:
    random.shuffle(data_all)
    test_ratio = int(len(data_all)*0.1)
    for dic in data_all[:test_ratio]:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/simcse_train_ljp.json"
with open(write_path, "w") as f:
    for dic in data_all[test_ratio:]:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")
print(len(data_all))   
        

100%|██████████| 100531/100531 [02:06<00:00, 797.63it/s]


70973


In [2]:
import pandas as pd
import json
import random
from tqdm import tqdm

with open("/root/data1/liang/self-correct-retriever/data/articles_base/刑法.json","r") as f:
    data_all = []
    lines = f.readlines()
    total_len  = 0
    for line in tqdm(lines):
        line = json.loads(line)
        total_len+=len(line["value"])
    print(total_len/len(lines))


100%|██████████| 582/582 [00:00<00:00, 244990.46it/s]

160.14948453608247
